# §7 Spine Ablation — Colab Runbook

This notebook runs the four-evening P3 vs P5 ablation laid out in `ABLATION_TIMELINE.md`. Each evening is a cell block you can run independently.

**Before you start:**
1. Open Colab → Runtime → Change runtime type → CPU is fine (the cost is API-side, not compute).
2. Click the key icon in the left sidebar, add a secret named `ANTHROPIC_API_KEY`, paste your key, toggle 'Notebook access' on.
3. Set a billing alert on the Anthropic console at `$50`, `$200`, `$400` before kicking off the full plan.

**Cost discipline:** mock pre-flight (free), then smoke (≈$5), then lean if smoke clean (≈$36-60 per model-spine pair), then full only if lean shows the predicted direction (total ≈$240-400).

## Setup (run once per Colab session)

In [ ]:
# Clone the repo into the Colab session
!git clone https://github.com/vasundras/agent-runtime-patterns.git /content/agent-runtime-patterns 2>/dev/null || (cd /content/agent-runtime-patterns && git pull)
%cd /content/agent-runtime-patterns

In [ ]:
# Install dependencies
!pip install -q 'anthropic>=0.39' --upgrade

In [ ]:
# Pull the API key from Colab secrets (never paste it in plaintext)
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-ant-'), 'API key not loaded from Colab secrets'
print('API key loaded.')

## Pre-flight — mock spine sanity check (free, ~5 sec)

Run this BEFORE Evening 1. Exercises both spine code paths with scripted responses (no API calls, no cost). If `cost_ledger.jsonl` populates here, the wiring is good and Evening 1 will produce real API calls.

**Exit gate:** ledger has at least 10 rows after this cell. If zero rows, there is a spine import error — check the output before paying for API calls.

In [ ]:
!python evals/run_eval.py --mock --spine p5 --limit 5 --k 1
!python evals/run_eval.py --mock --spine p3 --limit 5 --k 1
!wc -l evals/results/cost_ledger.jsonl

In [ ]:
# Reset the ledger and mock JSONL files before the real run so cost summary is clean.
!rm -f evals/results/cost_ledger.jsonl
!rm -f evals/results/p5_*_n5_k1.jsonl evals/results/p3_*_n5_k1.jsonl

## Evening 1 — Smoke test (≈$5, ~5 min)

**Goal:** prove the harness round-trips to the real Anthropic API and emits well-formed JSON. 5 scenarios, k=1, P5 only, Sonnet 4.6.

**Exit gate:** one JSON line per scenario in `evals/results/p5_*_n5_k1.jsonl`, no harness exceptions, `cost_ledger.jsonl` populated with non-zero token counts.

In [ ]:
!python evals/run_eval.py --live --spine p5 --model claude-sonnet-4-6 --limit 5 --k 1

In [ ]:
# Sanity check: did we actually call the API?
!head -3 evals/results/cost_ledger.jsonl && echo '---' && wc -l evals/results/cost_ledger.jsonl

## Evening 2 — Lean run, Sonnet 4.6 (≈$36-60, ~1.5 hours)

**Goal:** pass^4 for both spines on Sonnet 4.6 across 30 scenarios. 240 conversations total.

**Exit gate (sanity, not the publishable result):** P3 and P5 should have similar pass^1 (within 0.05). If they differ wildly, there is a bug in one spine, not a real signal.

In [ ]:
!python evals/run_eval.py --live --spine p5 --model claude-sonnet-4-6 --limit 30 --k 4
!python evals/run_eval.py --live --spine p3 --model claude-sonnet-4-6 --limit 30 --k 4

## Evening 3 — Lean run, Sonnet 4.5 (≈$72-120, ~3 hours)

**Goal:** same 30 scenarios on Sonnet 4.5. This is where drift becomes visible.

**Exit gate (the publishable signal):** does P3 show |Δpass^k| ≥ 0.10 while P5 shows |Δpass^k| ≤ 0.05?
- **Yes:** hypothesis holds at lean N. Proceed to Evening 4.
- **No (small Δ on both):** write up honestly. The drift effect is smaller than expected.
- **Reversed (P5 drifts more):** falsifying result. Report it. Update §4.2.

If you are using `claude-sonnet-4-5` and `claude-sonnet-4-6` as the version pair, make sure both identifiers are valid in the Anthropic console before kicking off — use a dated revision (e.g. `claude-sonnet-4-5-20251022`) if your account requires it.

In [ ]:
!python evals/run_eval.py --live --spine p5 --model claude-sonnet-4-5 --limit 30 --k 4
!python evals/run_eval.py --live --spine p3 --model claude-sonnet-4-5 --limit 30 --k 4

In [ ]:
# Decision gate — look at this before deciding whether to run Evening 4.
!python evals/analyze.py | head -40

## Evening 4 — Full run (≈$130-240, ~5 hours)

**Run only if Evening 3 supported the hypothesis.** Replaces lean N=30 with full N=100 on both spines and both models.

Total runtime depends on Anthropic's rate limits. Expect 4-6 hours wall clock for 1,600 conversations.

In [ ]:
!python evals/run_eval.py --live --spine p5 --model claude-sonnet-4-6 --limit 100 --k 4
!python evals/run_eval.py --live --spine p3 --model claude-sonnet-4-6 --limit 100 --k 4
!python evals/run_eval.py --live --spine p5 --model claude-sonnet-4-5 --limit 100 --k 4
!python evals/run_eval.py --live --spine p3 --model claude-sonnet-4-5 --limit 100 --k 4

## Analysis — produces the table that goes into §7

In [ ]:
!python evals/analyze.py

In [ ]:
# Display the final summary table
from IPython.display import Markdown
Markdown(open('evals/results/summary.md').read())

## Save results back to Drive (so you do not lose them when Colab disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/agent-runtime-patterns-results'
!cp -r evals/results/* '/content/drive/MyDrive/agent-runtime-patterns-results/'
!ls -la '/content/drive/MyDrive/agent-runtime-patterns-results/'